# 多功能 Agent 智能助手

支持功能：
- 🌤️ 天气查询
- 🔢 数学计算
- ⏰ 时间查询
- 💱 货币转换
- 🔍 信息检索

In [3]:
import os
import math
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 创建 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0
)
print("LLM 初始化完成")

LLM 初始化完成


## 1. 定义工具

In [6]:
# ==================== 天气查询 ====================
@tool
def get_weather(city: str) -> str:
    """查询指定城市的当前天气信息
    Args:
        city: 城市名称，如 "北京"、"上海"
    """
    # 模拟天气数据（实际应用可接入天气API）
    weather_data = {
        "北京": {"天气": "晴", "温度": "25°C", "湿度": "40%", "风力": "北风3级"},
        "上海": {"天气": "多云", "温度": "22°C", "湿度": "65%", "风力": "东风2级"},
        "广州": {"天气": "阵雨", "温度": "28°C", "湿度": "80%", "风力": "南风2级"},
        "深圳": {"天气": "晴转多云", "温度": "27°C", "湿度": "70%", "风力": "东南风3级"},
        "成都": {"天气": "阴", "温度": "20°C", "湿度": "75%", "风力": "微风"},
    }
    
    if city in weather_data:
        w = weather_data[city]
        return f"{city}天气：{w['天气']}，温度{w['温度']}，湿度{w['湿度']}，{w['风力']}"
    return f"暂无{city}的天气数据，支持的城市：{', '.join(weather_data.keys())}"

# ==================== 数学计算 ====================
@tool
def calculate(expression: str) -> str:
    """计算数学表达式，支持加减乘除、幂运算、三角函数等
    Args:
        expression: 数学表达式，如 "2+3*4"、"sqrt(16)"、"sin(3.14/2)"
    """
    # 安全的数学函数
    safe_dict = {
        "abs": abs, "round": round, "pow": pow,
        "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
        "tan": math.tan, "log": math.log, "log10": math.log10,
        "pi": math.pi, "e": math.e, "ceil": math.ceil, "floor": math.floor,
    }
    try:
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

# ==================== 时间查询 ====================
@tool
def get_current_time(timezone: str = "local") -> str:
    """获取当前时间和日期
    Args:
        timezone: 时区，"local"表示本地时间，"utc"表示UTC时间
    """
    if timezone.lower() == "utc":
        now = datetime.utcnow()
        return f"UTC时间：{now.strftime('%Y-%m-%d %H:%M:%S')}"
    else:
        now = datetime.now()
        weekdays = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
        return f"本地时间：{now.strftime('%Y-%m-%d %H:%M:%S')} {weekdays[now.weekday()]}"

# ==================== 货币转换 ====================
@tool
def currency_convert(amount: float, from_currency: str, to_currency: str) -> str:
    """货币转换（汇率为模拟数据）
    Args:
        amount: 金额
        from_currency: 源货币代码，如 "USD"、"CNY"、"EUR"、"JPY"
        to_currency: 目标货币代码
    """
    # 模拟汇率（实际应用可接入汇率API）
    rates = {
        "USD": {"CNY": 7.24, "EUR": 0.92, "JPY": 149.5, "GBP": 0.79},
        "CNY": {"USD": 0.138, "EUR": 0.127, "JPY": 20.65, "GBP": 0.109},
        "EUR": {"USD": 1.087, "CNY": 7.87, "JPY": 162.5, "GBP": 0.859},
        "JPY": {"USD": 0.0067, "CNY": 0.048, "EUR": 0.0062, "GBP": 0.0053},
        "GBP": {"USD": 1.265, "CNY": 9.17, "EUR": 1.164, "JPY": 188.5},
    }
    
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    
    if from_currency == to_currency:
        return f"{amount} {from_currency} = {amount} {to_currency}"
    
    if from_currency in rates and to_currency in rates[from_currency]:
        rate = rates[from_currency][to_currency]
        result = round(amount * rate, 2)
        return f"{amount} {from_currency} = {result} {to_currency}（汇率：1 {from_currency} = {rate} {to_currency}）"
    
    supported = list(rates.keys())
    return f"不支持的货币，支持：{', '.join(supported)}"

# ==================== 信息检索 ====================
@tool
def search_knowledge(query: str) -> str:
    """检索知识库中的信息，回答常识性问题
    Args:
        query: 查询问题
    """
    # 模拟知识库（实际应用可接入搜索引擎或知识图谱）
    knowledge_base = {
        "python": "Python是一种高级编程语言，由Guido van Rossum于1991年创建。特点：简洁易读、跨平台、丰富的库生态。",
        "langchain": "LangChain是一个用于构建LLM应用的框架，支持链式调用、Agent、RAG等模式。",
        "openai": "OpenAI是人工智能研究公司，开发了GPT系列模型，总部位于旧金山。",
        "机器学习": "机器学习是AI的分支，让计算机从数据中学习规律而无需显式编程。主要类型：监督学习、无监督学习、强化学习。",
        "太阳系": "太阳系包含8大行星：水星、金星、地球、火星、木星、土星、天王星、海王星。",
    }
    
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return value
    
    # return f"未找到关于"{query}"的信息，知识库包含：Python、LangChain、OpenAI、机器学习、太阳系等主题。"

# 收集所有工具
tools = [get_weather, calculate, get_current_time, currency_convert, search_knowledge]
print(f"已注册 {len(tools)} 个工具：{[t.name for t in tools]}")

已注册 5 个工具：['get_weather', 'calculate', 'get_current_time', 'currency_convert', 'search_knowledge']


## 2. 创建 Agent

In [7]:
# 创建多功能 Agent
agent = create_agent(llm, tools)
print("多功能 Agent 创建成功！")

多功能 Agent 创建成功！


## 3. 测试各项功能

In [27]:
def ask_agent(question: str):
    """向 Agent 提问并打印结果"""
    rprint(f"\n[bold blue]问题:[/bold blue] {question}")
    rprint("-" * 50)

    result = agent.invoke({"messages": [("human", question)]})

    # 打印执行过程
    for msg in result["messages"]:
        # rprint(msg, msg.type )
        if msg.type == "ai" and msg.tool_calls:
            for tc in msg.tool_calls:
                rprint(f"  [cyan]调用工具:[/cyan] {tc['name']}({tc['args']})")
        elif msg.type == "tool":
            rprint(f"  [green]工具结果:[/green] {msg.content}")

    # 最终回答
    final_answer = result["messages"][-1].content
    rprint(f"\n[bold yellow]回答:[/bold yellow] {final_answer}")
    return final_answer

In [24]:
# 测试天气查询
ask_agent("北京和上海今天天气怎么样？")

问题: 北京和上海今天天气怎么样？

--------------------------------------------------

调用工具: get_weather({'city': '北京'})

调用工具: get_weather({'city': '上海'})

工具结果: 北京天气：晴，温度25°C，湿度40%，北风3级

工具结果: 上海天气：多云，温度22°C，湿度65%，东风2级

回答: 你好！我来为你介绍一下北京和上海今天的天气情况：

**北京** ☀️
- 天气：晴
- 温度：25°C
- 湿度：40%
- 风向：北风3级

**上海** ☁️
- 天气：多云
- 温度：22°C
- 湿度：65%
- 风向：东风2级

从天气来看，北京今天阳光明媚，比较干燥；上海则是多云天气，湿度稍高一些。两地温度都很舒适，适合外出活动呢！

'你好！我来为你介绍一下北京和上海今天的天气情况：\n\n**北京** ☀️\n- 天气：晴\n- 温度：25°C\n- 湿度：40%\n- 风向：北风3级\n\n**上海** ☁️\n- 天气：多云\n- 温度：22°C\n- 湿度：65%\n- 风向：东风2级\n\n从天气来看，北京今天阳光明媚，比较干燥；上海则是多云天气，湿度稍高一些。两地温度都很舒适，适合外出活动呢！'

In [25]:
# 测试数学计算
ask_agent("计算 (15 + 27) * 3 - sqrt(144) 的结果")

问题: 计算 (15 + 27) * 3 - sqrt(144) 的结果

--------------------------------------------------

调用工具: calculate({'expression': '(15 + 27) * 3 - sqrt(144)'})

工具结果: (15 + 27) * 3 - sqrt(144) = 114.0

回答: 计算结果是 **114**。

计算过程如下：
1. 先算括号：15 + 27 = 42
2. 再乘3：42 × 3 = 126
3. 然后开平方：√144 = 12
4. 最后相减：126 - 12 = 114

所以，(15 + 27) × 3 - √144 = 114。

'计算结果是 **114**。\n\n计算过程如下：\n1. 先算括号：15 + 27 = 42\n2. 再乘3：42 × 3 = 126\n3. 然后开平方：√144 = 12\n4. 最后相减：126 - 12 = 114\n\n所以，(15 + 27) × 3 - √144 = 114。'

In [26]:
# 测试时间查询
ask_agent("现在几点了？今天星期几？")

问题: 现在几点了？今天星期几？

--------------------------------------------------

调用工具: get_current_time({'timezone': 'local'})

工具结果: 本地时间：2026-07-05 20:39:18 星期日

回答: 现在是 **2026年7月5日，星期日，晚上20点39分**。

今天是星期天，周末还没结束呢！希望你正享受愉快的休息时光。😊

'现在是 **2026年7月5日，星期日，晚上20点39分**。\n\n今天是星期天，周末还没结束呢！希望你正享受愉快的休息时光。😊'

In [ ]:
# 测试货币转换
ask_agent("100美元等于多少人民币？")

In [ ]:
# 测试信息检索
ask_agent("什么是Python？")

## 4. 复合问题测试

Agent 可以自动组合多个工具完成复杂任务

In [ ]:
# 复合问题：需要调用多个工具
ask_agent("现在几点了？如果现在是下午3点，把100美元换成人民币，然后告诉我换成人民币后能买几杯30元的咖啡？")

问题: 现在几点了？如果现在是下午3点，把100美元换成人民币，然后告诉我换成人民币后能买几杯30元的咖啡？

--------------------------------------------------

调用工具: get_current_time({'timezone': 'local'})

工具结果: 本地时间：2026-07-05 20:44:16 星期日

调用工具: currency_convert({'amount': 100, 'from_currency': 'USD', 'to_currency': 'CNY'})

...

调用工具: calculate({'expression': '724 / 30'})

工具结果: 724 / 30 = 24.133333333333333

回答: 当前时间是 **晚上8点44分**，并不是下午3点。不过，我还是可以帮你完成货币转换和计算：

1. **100美元兑换人民币**：按当前汇率（1 USD = 7.24 CNY），100美元可兑换 **724元人民币**。
2. **能买几杯30元的咖啡**：724 ÷ 30 ≈ 24.13，所以最多能买 **24杯**，还会剩余约4元。

如果时间真的是下午3点，这个计算结果也是一样的。需要我帮你做其他换算吗？

'当前时间是 **晚上8点44分**，并不是下午3点。不过，我还是可以帮你完成货币转换和计算：\n\n1. **100美元兑换人民币**：按当前汇率（1 USD = 7.24 CNY），100美元可兑换 **724元人民币**。\n2. **能买几杯30元的咖啡**：724 ÷ 30 ≈ 24.13，所以最多能买 **24杯**，还会剩余约4元。\n\n如果时间真的是下午3点，这个计算结果也是一样的。需要我帮你做其他换算吗？'

: 

In [ ]:
# 复合问题：天气+计算
ask_agent("北京今天多少度？如果温度乘以2再加10等于多少？")

## 5. 流式输出模式

In [ ]:
# 流式输出展示 Agent 执行过程
question = "帮我查一下上海天气，然后计算2的10次方是多少"
rprint(f"[bold blue]问题:[/bold blue] {question}")
rprint("=" * 50)

for i, state in enumerate(
    agent.stream({"messages": [("human", question)]}, stream_mode="values")
):
    last_msg = state["messages"][-1]
    
    if last_msg.type == "ai":
        if last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                rprint(f"\n[bold cyan]步骤 {i+1} - 调用工具:[/bold cyan] {tc['name']}")
                rprint(f"  参数: {tc['args']}")
        elif last_msg.content:
            rprint(f"\n[bold yellow]最终回答:[/bold yellow]")
            rprint(last_msg.content)
    elif last_msg.type == "tool":
        rprint(f"  [green]结果:[/green] {last_msg.content}")

## 6. 带记忆的对话

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 创建带记忆的 Agent
checkpointer = MemorySaver()
agent_with_memory = create_agent(llm, tools, checkpointer=checkpointer)

# 使用 thread_id 维护对话状态
config = {"configurable": {"thread_id": "user-1"}}

# 第一轮对话
rprint("[bold]第一轮对话[/bold]")
result = agent_with_memory.invoke(
    {"messages": [("human", "北京天气怎么样？")]},
    config=config
)
rprint(f"回答: {result['messages'][-1].content}\n")

# 第二轮对话（记住上下文）
rprint("[bold]第二轮对话[/bold]")
result = agent_with_memory.invoke(
    {"messages": [("human", "那上海呢？")]},
    config=config
)
rprint(f"回答: {result['messages'][-1].content}")

## 总结

### 工具列表
| 工具 | 功能 | 示例 |
|------|------|------|
| `get_weather` | 天气查询 | "北京天气怎么样" |
| `calculate` | 数学计算 | "计算 sqrt(144)" |
| `get_current_time` | 时间查询 | "现在几点" |
| `currency_convert` | 货币转换 | "100美元兑人民币" |
| `search_knowledge` | 信息检索 | "什么是Python" |

### Agent 特点
- **自动决策**: 根据问题自动选择合适的工具
- **多工具组合**: 可串联多个工具完成复杂任务
- **流式输出**: 支持实时查看执行过程
- **对话记忆**: 通过 checkpointer 维护上下文